In [1]:
import os
import numpy as np
import scipy.io
import pandas as pd
from io import StringIO

# ================= CONFIGURAÇÕES =================

DIRETORIO_RAIZ = r"C:\Users\batis\OneDrive\Área de Trabalho\BCI_MAT\transfer_MAT"
CSV_SAIDA = r"C:\Users\batis\OneDrive\Área de Trabalho\Bci_py_unity\ARQUVOOPENBCI_16CANAIS.csv"
LABELS_SAIDA = r"labels_ordem.txt"

T_ANTES = 0.0
T_DEPOIS = 1.0

# classes no .mat -> 0: repouso, 1: esquerda, 2: direita
MAPA_CLASSES = {1: 0, 2: 1, 3: 2}

# Lista original de 21 canais do dataset
CANAIS_ORIGINAIS = ['Fp1','Fp2','F3','F4','C3','C4','P3','P4','O1','O2',
                    'A1','A2','F7','F8','T3','T4','T5','T6','Fz','Cz','Pz']

# 16 canais que você quer extrair
CANAIS_16 = ['Fp1','F7','F3','T3','C3','T5','P3','O1',
             'Fp2','F4','F8','C4','T4','P4','T6','O2']

# converte nomes para índices dentro de CANAIS_ORIGINAIS
INDICES_16 = [CANAIS_ORIGINAIS.index(ch) for ch in CANAIS_16]

# ===================================================


def encontrar_indices_automaticamente(struct):
    eeg_data = None
    marker_channel = None
    sfreq = 200  # fallback

    for item in struct:
        if isinstance(item, (int, float, np.integer, np.floating)):
            if 20 <= item <= 5000:
                sfreq = int(item)
        elif isinstance(item, np.ndarray):
            if item.size == 0:
                continue
            shape = item.shape

            # EEG: matriz 2D
            if len(shape) == 2:
                # checa 21 ou 22 canais
                if (shape[0] in [21,22] and shape[1] > 500):
                    eeg_data = item
                elif (shape[1] in [21,22] and shape[0] > 500):
                    eeg_data = item.T

            # marcador
            if (len(shape) == 1 and shape[0] > 500) or \
               (len(shape) == 2 and (shape[0]==1 or shape[1]==1) and item.size > 500):
                flat = item.flatten()
                # assume valores discretos baixos
                if np.nanmax(np.abs(flat)) < 100:
                    marker_channel = flat

    return eeg_data, marker_channel, sfreq


def extrair_epocas_mat(diretorio):
    """
    Extrai todas as épocas de todos os .mat dentro do diretório
    retornando uma lista de (época, label).
    """
    epocas = []
    arquivos = [f for f in os.listdir(diretorio) if f.lower().endswith('.mat')]

    for nome in arquivos:
        caminho = os.path.join(diretorio, nome)
        try:
            mat = scipy.io.loadmat(caminho, squeeze_me=True)
        except Exception as e:
            print(f"Erro lendo {nome}: {e}")
            continue

        if 'o' not in mat:
            print(f"Arquivo {nome} sem struct 'o'. Pulando.")
            continue

        struct = mat['o']
        # às vezes vem como array de objetos
        try:
            struct = struct.item()
        except:
            pass

        eeg_data, marker_channel, sfreq = encontrar_indices_automaticamente(struct)
        if eeg_data is None or marker_channel is None:
            print(f"Não identifiquei EEG ou marcador em {nome}. Pulando.")
            continue

        # limita tamanho temporal
        minimo = min(eeg_data.shape[1], len(marker_channel))
        eeg_data = eeg_data[:, :minimo]
        marker_channel = marker_channel[:minimo]

        # agora filtra só os 16 canais desejados
        eeg_data = eeg_data[INDICES_16, :]

        n_antes = int(T_ANTES * sfreq)
        n_depois = int(T_DEPOIS * sfreq)
        total = n_antes + n_depois

        diffs = np.diff(marker_channel, prepend=marker_channel[0])
        eventos = np.where(diffs != 0)[0]

        for idx in eventos:
            classe = int(marker_channel[idx])
            if classe in MAPA_CLASSES:
                ini = idx - n_antes
                fim = idx + n_depois
                if ini >= 0 and fim <= eeg_data.shape[1]:
                    ep = eeg_data[:, ini:fim]
                    if ep.shape[1] == total:
                        epocas.append((ep, MAPA_CLASSES[classe]))

        print(f"{nome} -> {len(epocas)} épocas extraídas até agora.")

    return epocas, sfreq


def salvar_csv_openbci(epocas, sfreq, caminho_csv):
    """
    Salva o CSV no formato OpenBCI clássico com
    cabeçalho e todas as amostras concatenadas.
    """
    n_ep = len(epocas)
    n_ch = len(CANAIS_16)
    n_t = epocas[0][0].shape[1]

    # reorganiza: (época, tempo, canais)
    dados = np.stack([e for e,_ in epocas], axis=0)
    dados = dados.transpose(0,2,1).reshape(n_ep*n_t, n_ch)

    df = pd.DataFrame(dados, columns=[f'EXG Channel {i}' for i in range(n_ch)])

    # Monta colunas padrão estilo OpenBCI
    df.insert(0, 'Sample Index', range(1, len(df)+1))
    df['Accel Channel 0'] = 0
    df['Accel Channel 1'] = 0
    df['Accel Channel 2'] = 0
    for i in range(7):
        df[f'Other_{i}'] = 0
    df['Analog Channel 0'] = 0
    df['Analog Channel 1'] = 0
    df['Analog Channel 2'] = 0

    # Timestamp
    timestamps = np.arange(len(df))/sfreq
    df['Timestamp'] = np.round(timestamps,6)
    df['Other'] = 0
    df['Timestamp (Formatted)'] = pd.to_timedelta(df['Timestamp'], unit='s').astype(str)

    # Escreve CSV com cabeçalho igual OpenBCI
    header = [
        '%OpenBCI Raw EXG Data\n',
        f'%Number of channels = {n_ch}\n',
        f'%Sample Rate = {sfreq} Hz\n',
        '%Board = OpenBCI_GUI$BoardCytonSerialDaisy\n'
    ]

    with open(caminho_csv, 'w', newline='') as f:
        f.writelines(header)
        df.to_csv(f, index=False)


def salvar_labels(epocas, caminho_txt):
    with open(caminho_txt, 'w') as f:
        for _, lbl in epocas:
            f.write(f"{lbl}\n")


# ================= EXECUÇÃO =================

epocas, sfreq = extrair_epocas_mat(DIRETORIO_RAIZ)
print(f"Total épocas extraídas: {len(epocas)}")

print("Salvando CSV OpenBCI…")
salvar_csv_openbci(epocas, sfreq, CSV_SAIDA)

print("Salvando labels…")
salvar_labels(epocas, LABELS_SAIDA)

print("CONCLUÍDO!")


CLASubjectA1601083StLRHand.mat -> 960 épocas extraídas até agora.
Total épocas extraídas: 960
Salvando CSV OpenBCI…
Salvando labels…
CONCLUÍDO!


In [1]:
import os
import numpy as np
import scipy.io
import pandas as pd

# ================= CONFIGURAÇÕES =================

DIRETORIO_RAIZ = r"C:\Users\batis\OneDrive\Área de Trabalho\BCI_MAT\transfer_MAT"
CSV_SAIDA = r"C:\Users\batis\OneDrive\Área de Trabalho\Bci_py_unity\ARQUVOOPENBCI_16CANAIS.csv"
LABELS_SAIDA = r"labels_ordem.txt"

T_ANTES = 0.0
T_DEPOIS = 1.0

# classes no .mat -> 0: repouso, 1: esquerda, 2: direita
MAPA_CLASSES = {1: 0, 2: 1, 3: 2}

# Lista original de 21 canais do dataset
CANAIS_ORIGINAIS = ['Fp1','Fp2','F3','F4','C3','C4','P3','P4','O1','O2',
                    'A1','A2','F7','F8','T3','T4','T5','T6','Fz','Cz','Pz']

# 16 canais que você quer extrair
CANAIS_16 = ['Fp1','F7','F3','T3','C3','T5','P3','O1',
             'Fp2','F4','F8','C4','T4','P4','T6','O2']

INDICES_16 = [CANAIS_ORIGINAIS.index(ch) for ch in CANAIS_16]

# ===================================================


def encontrar_indices_automaticamente(struct):
    eeg_data = None
    marker_channel = None
    sfreq = 200  # fallback

    for item in struct:
        if isinstance(item, (int, float, np.integer, np.floating)):
            if 20 <= item <= 5000:
                sfreq = int(item)

        elif isinstance(item, np.ndarray):
            if item.size == 0:
                continue

            shape = item.shape

            # EEG
            if len(shape) == 2:
                if (shape[0] in [21,22] and shape[1] > 500):
                    eeg_data = item
                elif (shape[1] in [21,22] and shape[0] > 500):
                    eeg_data = item.T

            # Marcador
            if (len(shape) == 1 and shape[0] > 500) or \
               (len(shape) == 2 and (shape[0]==1 or shape[1]==1) and item.size > 500):
                flat = item.flatten()
                if np.nanmax(np.abs(flat)) < 100:
                    marker_channel = flat

    return eeg_data, marker_channel, sfreq


def extrair_epocas_mat(diretorio):
    epocas = []
    arquivos = [f for f in os.listdir(diretorio) if f.lower().endswith('.mat')]

    for nome in arquivos:
        caminho = os.path.join(diretorio, nome)
        try:
            mat = scipy.io.loadmat(caminho, squeeze_me=True)
        except Exception as e:
            print(f"Erro lendo {nome}: {e}")
            continue

        if 'o' not in mat:
            print(f"Arquivo {nome} sem struct 'o'. Pulando.")
            continue

        struct = mat['o']
        try:
            struct = struct.item()
        except:
            pass

        eeg_data, marker_channel, sfreq = encontrar_indices_automaticamente(struct)
        if eeg_data is None or marker_channel is None:
            print(f"Não identifiquei EEG ou marcador em {nome}. Pulando.")
            continue

        minimo = min(eeg_data.shape[1], len(marker_channel))
        eeg_data = eeg_data[:, :minimo]
        marker_channel = marker_channel[:minimo]

        eeg_data = eeg_data[INDICES_16, :]

        n_antes = int(T_ANTES * sfreq)
        n_depois = int(T_DEPOIS * sfreq)
        total = n_antes + n_depois

        diffs = np.diff(marker_channel, prepend=marker_channel[0])
        eventos = np.where(diffs != 0)[0]

        for idx in eventos:
            classe = int(marker_channel[idx])
            if classe in MAPA_CLASSES:
                ini = idx - n_antes
                fim = idx + n_depois
                if ini >= 0 and fim <= eeg_data.shape[1]:
                    ep = eeg_data[:, ini:fim]
                    if ep.shape[1] == total:
                        epocas.append((ep, MAPA_CLASSES[classe]))

        print(f"{nome} -> {len(epocas)} épocas acumuladas.")

    return epocas, sfreq


def salvar_csv_openbci(epocas, sfreq, caminho_csv):
    n_ep = len(epocas)
    n_ch = len(CANAIS_16)
    n_t = epocas[0][0].shape[1]

    dados = np.stack([e for e,_ in epocas], axis=0)
    dados = dados.transpose(0,2,1).reshape(n_ep*n_t, n_ch)

    # ===== LINHAS ZERADAS ANTES DO SINAL =====
    LINHAS_ZERADAS_INICIAIS = sfreq * 2  # 2 segundos de zeros
    zeros = np.zeros((LINHAS_ZERADAS_INICIAIS, n_ch))
    dados = np.vstack([zeros, dados])
    # =========================================

    df = pd.DataFrame(dados, columns=[f'EXG Channel {i}' for i in range(n_ch)])
    df.insert(0, 'Sample Index', range(1, len(df)+1))

    df['Accel Channel 0'] = 0
    df['Accel Channel 1'] = 0
    df['Accel Channel 2'] = 0
    for i in range(7):
        df[f'Other_{i}'] = 0
    df['Analog Channel 0'] = 0
    df['Analog Channel 1'] = 0
    df['Analog Channel 2'] = 0

    timestamps = np.arange(len(df)) / sfreq
    df['Timestamp'] = np.round(timestamps, 6)
    df['Other'] = 0
    df['Timestamp (Formatted)'] = pd.to_timedelta(df['Timestamp'], unit='s').astype(str)

    header = [
        '%OpenBCI Raw EXG Data\n',
        f'%Number of channels = {n_ch}\n',
        f'%Sample Rate = {sfreq} Hz\n',
        '%Board = OpenBCI_GUI$BoardCytonSerialDaisy\n'
    ]

    with open(caminho_csv, 'w', newline='') as f:
        f.writelines(header)
        df.to_csv(f, index=False)


def salvar_labels(epocas, caminho_txt):
    with open(caminho_txt, 'w') as f:
        for _, lbl in epocas:
            f.write(f"{lbl}\n")


# ================= EXECUÇÃO =================

epocas, sfreq = extrair_epocas_mat(DIRETORIO_RAIZ)
print(f"Total épocas extraídas: {len(epocas)}")

print("Salvando CSV OpenBCI…")
salvar_csv_openbci(epocas, sfreq, CSV_SAIDA)

print("Salvando labels…")
salvar_labels(epocas, LABELS_SAIDA)

print("CONCLUÍDO!")


CLASubjectA1601083StLRHand.mat -> 960 épocas acumuladas.
Total épocas extraídas: 960
Salvando CSV OpenBCI…
Salvando labels…
CONCLUÍDO!
